In [1]:
# spark.stop()

In [2]:
import os
from pyspark.sql import SparkSession, types as t, functions as F
from pyspark.sql.types import StringType, FloatType, IntegerType

# https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar

spark = (
    SparkSession
    .builder
    # .master("spark://spark-master:7077")
    .appName("Testing Transformations")
    .config("spark.jars", "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar") # GCS Connector
    .getOrCreate()
)

# Google Cloud Service Account Credentials
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile",os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))

spark

your 131072x1 screen size is bogus. expect trouble
25/05/28 00:53:13 WARN Utils: Your hostname, Trydex resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/05/28 00:53:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/28 00:53:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
bucket='gs://zoomcamp-454219-ade-pipeline/data/pq/'
schema = 'reaction'
year = 2025
event = "drug-event-part-1-of-34.parquet" # 2025
# year = 2004
# event = "drug-event-part-1-of-20.parquet" # 2004

df = (
    spark
    .read
    .parquet(bucket+f'{schema}/{year}/{event}')
    )
print(f"Count: {df.count()}")
# print(df.printSchema())
# patient.show()

Count: 60118


In [4]:
class Reaction:
    def __init__(self, df):
        self.df = df
        
    def get_df(self):
        return self.df
    
    def cast(self):
        self.df = (
            self.df
            .withColumn("patientid", F.col("patientid").cast(StringType()))
            .withColumn("reactionmeddrapt", F.col("reactionmeddrapt").cast(StringType()))
            .withColumn("reactionoutcome", F.col("reactionoutcome").cast(StringType()))
        )

    def transform(self):
        self.df = (
            self.df
            .withColumn(
                "reactionoutcome",
                (
                    F
                    .when(F.col("reactionoutcome") == '1', "Recovered/resolved")
                    .when(F.col("reactionoutcome") == '2', "Recovering/resolving")
                    .when(F.col("reactionoutcome") == '3', "Not recovered/not resolved")
                    .when(F.col("reactionoutcome") == '4', "Recovered/resolved with sequelae (consequent health issues)")
                    .when(F.col("reactionoutcome") == '5', "Fatal")
                    .when(F.col("reactionoutcome") == '6', "Unknown")
                    .otherwise(None)
                ).cast(StringType())
            )
        )

        # Handle null
        self.handle_null()
    
    def handle_null(self):
        fillna_dict = {
            'reactionmeddrapt' : 'Unspecified',
            'reactionoutcome' : 'Unknown',
        }

        self.df = self.df.fillna(fillna_dict)

In [5]:
r = Reaction(df)
r.cast()
r.transform()

In [6]:
r.get_df().columns

['patientid', 'reactionmeddrapt', 'reactionoutcome']

In [102]:
df_processed = r.get_df()
df_processed

DataFrame[patientid: string, reactionmeddrapt: string, reactionoutcome: string]

In [103]:
rdf = df_processed.toPandas()
rdf.shape

(60118, 3)

In [104]:
rdf.isna().sum()

patientid           0
reactionmeddrapt    0
reactionoutcome     0
dtype: int64

In [123]:
rdf.sample(n=10)

,patientid,reactionmeddrapt,reactionoutcome
31276,fce68f66-e974-406f-a76f-41f72425c6e6,Pain,Recovered/resolved
10200,f25385d5-62d2-4591-8e16-120e808f3bbb,Fatigue,Not recovered/not resolved
44106,7d34cb2c-6df9-4c9e-80d0-50e4b57946ee,Chest discomfort,Recovered/resolved
42617,de6a53b8-8ba2-4178-b58c-14c3def49ceb,Drug dose omission by device,Unknown
58708,7ffe7fba-905c-4844-b7fe-390ac1176f08,Depression,Unknown
48331,6650f7c1-6f4b-4fc3-b792-5d8bc9685381,Musculoskeletal discomfort,Not recovered/not resolved
12840,363ea364-6300-4a13-8b5a-bbef5fceb79b,COVID-19,Recovered/resolved
16576,0ff0f135-d944-474d-9f44-87b7b251c80b,Pulmonary fibrosis,Fatal
44379,a8b3a0a4-1724-4d21-b49e-cd00e43258b9,Impaired quality of life,Unknown
30580,eff66519-5cbf-4f54-bbd1-bf2ffe160055,Accident,Fatal


In [79]:
rdf.reactionmeddrapt.nunique()

3818

In [78]:
rdf[~rdf.reactionoutcome.isna()].sample(n=1)

,patientid,reactionmeddrapt,reactionoutcome
27943,227dacd6-3202-44bd-9337-03be2bbd5771,Fall,Recovered/resolved


In [93]:
rdf.reactionmeddrapt.sample(n=20).to_frame()

,reactionmeddrapt
5959,MENTAL STATUS CHANGES
33945,CD4 LYMPHOCYTES DECREASED
14282,BLOODY DISCHARGE
3620,HYPOGLYCAEMIA
20770,PREGNANCY
34599,SKIN HYPOPIGMENTATION
28333,NEUTROPENIA
1240,GASTRIC HAEMORRHAGE
10809,CONVULSION
35986,LIBIDO DECREASED
